# 02 Mouse Data Products for Comparison

## Overview

This notebook builds annotated, all-gene mouse AnnData objects for downstream
cross-species and condition-specific analyses.

## Scientific and Technical Purpose

- Load scAR ambient-RNA-corrected counts for all samples (all genes retained)
- Filter to typed cells and merge with cell-type annotations from the master barcode reference (NB01)
- Attach sample ID, experimental condition, and all taxonomic annotations
- Export a canonical all-cells object for exploratory analysis (notebook 06+)
- Export microglia in both mouse gene space and as a human-gene-name-translated object (NB03)
- Export an astrocyte-specific subset for astrocyte branch workflows

## Upstream Dependencies

- `00_Ambient_RNA_correction.ipynb` outputs:
  - `Analysis/scAR/h5ad/*_scar_corrected.h5ad` (counts only; no annotations)
- `01_Ingest.ipynb` outputs:
  - `Results/Tables/all_cell_barcodes_nb01.csv` (master reference: union of all typed cells + class/subclass/cluster annotations)

## Primary Outputs

- `adatas/brain_allcells_allgenes.h5ad`          — all typed cells, all mouse genes, all annotations
- `adatas/brain_microglia_allgenes.h5ad`          — microglia subset, mouse gene space
- `adatas/mouse_microglia_humanized_nb02.h5ad`    — microglia subset, human gene names (for NB03 cross-species)
- `adatas/brain_astrocytes_allgenes.h5ad`         — astrocyte subset
- `Microglia_analysis/mouse_human_orthologs_nb02.csv` — ortholog lookup table

## Quality and Reproducibility Standards

- **Cell selection**: Uses union of all typed cells from NB01 (not re-typed or filtered here)
- **Annotations**: Direct pass-through from NB01; no manual curation
- **Gene space preservation**: Raw counts layer + normalized log-transform layer; humanized microglia retains mouse_gene column for traceability
- **Metadata flow**: All sample, condition, and taxonomy columns propagated from NB01 master reference

## Runtime and Compute

Typical runtime is 3–10 minutes depending on storage I/O.

# Table of Contents

1. [Imports and Environment](#imports)

2. [Configuration and Paths](#configuration)

3. [Data Loading](#data-loading)
   - Sample manifest from scAR h5ad files
   - Master cell reference from notebook 01

4. [Build Annotated All-Cells AnnData](#all-cells)
   - Per-sample load, filter, and annotate
   - Concatenate → normalize → save

5. [Subset Exports](#subsets)
   - Microglia (for NB03)
   - Astrocytes

6. [Production Checks](#checks)

7. [Summary](#summary)

In [1]:
# Consolidated imports (PEP8: stdlib, third-party, local)
import os
import time
import copy
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy.sparse
import matplotlib.pyplot as plt
import seaborn as sns
from mousipy import translate

# Local imports will be added after BASE_DIR is set

In [2]:
# Version logging for reproducibility
import sys
print(f"Python version: {sys.version}")
print(f"Scanpy: {sc.__version__}")
print(f"Anndata: {ad.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Numpy: {np.__version__}")
print(f"Seaborn: {sns.__version__}")
import matplotlib
print(f"Matplotlib: {matplotlib.__version__}")

Python version: 3.10.13 (main, Feb  2 2026, 16:27:42) [GCC 13.3.0]
Scanpy: 1.11.5
Anndata: 0.11.4
Pandas: 2.3.3
Numpy: 2.2.6
Seaborn: 0.13.2
Matplotlib: 3.10.0


/scratch/ipykernel_156147/3639955968.py:4: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(f"Scanpy: {sc.__version__}")


In [3]:
# Configuration: paths, reproducibility, and runtime options
NOTEBOOKS_DIR = Path.cwd().resolve()
if NOTEBOOKS_DIR.name != "Notebooks":
    NOTEBOOKS_DIR = (NOTEBOOKS_DIR / "Notebooks").resolve()
ANALYSIS_DIR = NOTEBOOKS_DIR.parent.resolve()
BASE_DIR = ANALYSIS_DIR.parent.resolve()
ADATA_DIR = ANALYSIS_DIR / "adatas"
MAPPING_DIR = ANALYSIS_DIR / "Mapping"
MICROGLIA_DIR = ANALYSIS_DIR / "Microglia_analysis"
ASTRO_DIR = ANALYSIS_DIR / "Astrocyte_analysis"
SCAR_H5AD_DIR = ANALYSIS_DIR / "scAR" / "h5ad"

UPSTREAM_NOTEBOOK_TAG = "nb01"
RANDOM_SEED = 42
TARGET_SUM = 1e4
OVERWRITE_OUTPUTS = True

# Set random seeds
np.random.seed(RANDOM_SEED)
import random
random.seed(RANDOM_SEED)

# Configure Scanpy settings
sc.settings.set_figure_params(dpi=100, dpi_save=300, figsize=(10, 8))
sc.settings.verbosity = 3

# Create output directories if they don't exist
for d in [ANALYSIS_DIR, ADATA_DIR, MAPPING_DIR, MICROGLIA_DIR, ASTRO_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Local imports (add parent dir to path for relative imports)
import sys
sys.path.append(str(ANALYSIS_DIR))
from utils import pp, plot_embedding

# Data Loading

In [4]:
# Build a deterministic sample manifest from notebook 00 scAR-corrected h5ad files
h5ad_files = sorted(SCAR_H5AD_DIR.glob("*_scar_corrected.h5ad"))
if not h5ad_files:
    raise FileNotFoundError(f"No scAR-corrected h5ad files found in: {SCAR_H5AD_DIR}")

data = [
    {
        "Sample": p.name.replace("_scar_corrected.h5ad", ""),
        "Path": str(p),
    }
    for p in h5ad_files
]

df = pd.DataFrame(data).sort_values("Sample").reset_index(drop=True)
if df["Sample"].duplicated().any():
    dupes = df.loc[df["Sample"].duplicated(), "Sample"].tolist()
    raise ValueError(f"Duplicate sample names found in manifest: {dupes}")

print(f"Discovered {len(df)} scAR-corrected samples")
display(df)

Discovered 6 scAR-corrected samples


,Sample,Path
0,Mock-1,/media/drive_c/Project_Brain_snRNAseq/Analysis...
1,Mock-2,/media/drive_c/Project_Brain_snRNAseq/Analysis...
2,Mock-3,/media/drive_c/Project_Brain_snRNAseq/Analysis...
3,OG-1,/media/drive_c/Project_Brain_snRNAseq/Analysis...
4,OG-2,/media/drive_c/Project_Brain_snRNAseq/Analysis...
5,OG-3,/media/drive_c/Project_Brain_snRNAseq/Analysis...


In [5]:
# Load the master cell reference produced by notebook 01.
# Contains every typed cell barcode plus sample ID, condition, and all annotation columns.
master_ref_path = (ANALYSIS_DIR / "Results" / "Tables" / f"all_cell_barcodes_{UPSTREAM_NOTEBOOK_TAG}.csv")
if not master_ref_path.exists():
    raise FileNotFoundError(
        f"Master reference not found: {master_ref_path}\n"
        "Run notebook 01 first to generate this file."
    )

master_ref = pd.read_csv(master_ref_path)

required_cols = ["cell_barcode", "Sample", "Condition", "class_name",
                 "class_broad", "subclass_name", "cluster_name"]
missing = [c for c in required_cols if c not in master_ref.columns]
if missing:
    raise KeyError(f"Master reference missing expected columns: {missing}")

print(f"Master reference: {len(master_ref):,} typed cells across "
      f"{master_ref['Sample'].nunique()} samples")
print(f"Samples: {sorted(master_ref['Sample'].unique())}")
print(f"Conditions: {master_ref['Condition'].value_counts().to_dict()}")
print(f"Cell types (class_broad): {master_ref['class_broad'].value_counts().to_dict()}")

Master reference: 50,894 typed cells across 6 samples
Samples: ['Mock-1', 'Mock-2', 'Mock-3', 'OG-1', 'OG-2', 'OG-3']
Conditions: {'OG': 25536, 'Mock': 25358}
Cell types (class_broad): {'Glut': 26357, 'GABA': 22258, 'Vascular': 1223, 'OPC': 399, 'Immune': 280, 'Astro': 249, 'Dopa': 99, 'Sero': 21, 'OEC': 8}


# Build Annotated All-Cells AnnData

In [6]:
# For each sample: load scAR-corrected h5ad, filter to typed cells from NB01,
# attach sample/condition/cell-type annotations, then concatenate into one object.
# All genes are retained (no HVG selection).

ANNOTATION_COLS = ["Sample", "Condition", "class_name", "class_broad",
                   "subclass_name", "cluster_name", "cluster_alias"]

per_sample_adatas = []
build_stats = []

for _, row in df.iterrows():
    sample_name = row["Sample"]
    file_path = Path(row["Path"])
    t0 = time.time()

    raw = ad.read_h5ad(file_path)
    raw.var_names_make_unique()
    n_input = raw.n_obs

    # Barcodes typed in NB01 for this specific sample
    sample_ref = master_ref[master_ref["Sample"] == sample_name].set_index("cell_barcode")
    kept = sample_ref.index.intersection(raw.obs_names)

    if len(kept) == 0:
        print(f"  WARNING: No typed barcodes found for {sample_name} — skipping")
        build_stats.append({
            "Sample": sample_name, "InputCells": n_input,
            "TypedCells": 0, "Genes": 0, "Status": "Skipped (no matching barcodes)",
        })
        del raw
        continue

    adata = raw[kept].copy()
    del raw  # free memory

    # Preserve raw ambient-corrected counts before any normalisation
    adata.layers["counts"] = adata.X.copy()

    # Attach all annotation columns from the master reference
    for col in ANNOTATION_COLS:
        if col in sample_ref.columns:
            adata.obs[col] = sample_ref.loc[kept, col].values

    elapsed = time.time() - t0
    per_sample_adatas.append(adata)
    build_stats.append({
        "Sample": sample_name,
        "InputCells": n_input,
        "TypedCells": adata.n_obs,
        "Genes": adata.n_vars,
        "Status": f"OK ({elapsed:.1f}s)",
    })
    print(f"  {sample_name}: {n_input:,} cells in → {adata.n_obs:,} typed cells × {adata.n_vars:,} genes ({elapsed:.1f}s)")

build_stats_df = pd.DataFrame(build_stats)
display(build_stats_df)

if not per_sample_adatas:
    raise RuntimeError(
        "No samples processed. Check that NB01 outputs exist and sample names "
        f"in the manifest match those in {master_ref_path}."
    )

# ── Concatenate ──────────────────────────────────────────────────────────────
all_cells_adata = ad.concat(per_sample_adatas, join="inner", merge="same")
all_cells_adata.obs_names_make_unique()
all_cells_adata.var_names_make_unique()
print(f"\nConcatenated: {all_cells_adata.n_obs:,} cells × {all_cells_adata.n_vars:,} genes")

# ── Normalize and log-transform ───────────────────────────────────────────────
# X = raw counts at this point; preserve in layer before modifying
sc.pp.normalize_total(all_cells_adata, target_sum=TARGET_SUM)
sc.pp.log1p(all_cells_adata)
all_cells_adata.layers["log_norm"] = all_cells_adata.X.copy()

print(f"obs columns: {list(all_cells_adata.obs.columns)}")

# ── Save all-cells AnnData ────────────────────────────────────────────────────
allcells_path = ADATA_DIR / "brain_allcells_allgenes.h5ad"
if allcells_path.exists() and not OVERWRITE_OUTPUTS:
    raise FileExistsError(f"Output exists and OVERWRITE_OUTPUTS=False: {allcells_path}")

all_cells_adata.uns["pipeline_metadata"] = {
    "pipeline": "02_mouse_prep_for_comparison",
    "artifact": "brain_allcells_allgenes",
    "upstream_tag": UPSTREAM_NOTEBOOK_TAG,
    "n_samples": len(per_sample_adatas),
    "n_cells": int(all_cells_adata.n_obs),
    "n_genes": int(all_cells_adata.n_vars),
}
all_cells_adata.write_h5ad(allcells_path, compression="gzip")
print(f"All-cells AnnData ready: {allcells_path} — {all_cells_adata.shape}")

  Mock-1: 13,988 cells in → 9,197 typed cells × 16,563 genes (1.3s)
  Mock-2: 16,120 cells in → 7,861 typed cells × 16,597 genes (1.4s)
  Mock-3: 14,415 cells in → 8,300 typed cells × 16,567 genes (1.4s)
  OG-1: 15,845 cells in → 9,828 typed cells × 16,526 genes (1.4s)
  OG-2: 14,282 cells in → 8,226 typed cells × 16,606 genes (1.4s)
  OG-3: 14,974 cells in → 7,482 typed cells × 16,609 genes (1.4s)


,Sample,InputCells,TypedCells,Genes,Status
0,Mock-1,13988,9197,16563,OK (1.3s)
1,Mock-2,16120,7861,16597,OK (1.4s)
2,Mock-3,14415,8300,16567,OK (1.4s)
3,OG-1,15845,9828,16526,OK (1.4s)
4,OG-2,14282,8226,16606,OK (1.4s)
5,OG-3,14974,7482,16609,OK (1.4s)



Concatenated: 50,894 cells × 16,033 genes
normalizing counts per cell
    finished (0:00:00)


... storing 'Sample' as categorical
... storing 'Condition' as categorical
... storing 'class_name' as categorical
... storing 'class_broad' as categorical
... storing 'subclass_name' as categorical
... storing 'cluster_name' as categorical


obs columns: ['n_genes', 'n_counts', 'Sample', 'Condition', 'class_name', 'class_broad', 'subclass_name', 'cluster_name', 'cluster_alias']
All-cells AnnData ready: /media/drive_c/Project_Brain_snRNAseq/Analysis/adatas/brain_allcells_allgenes.h5ad — (50894, 16033)


# Subset Exports

Produce cell-type-specific subsets from the all-cells object for downstream notebooks.

In [7]:
# ── Microglia subset (class_broad == "Immune") ────────────────────────────────
# Primary strategy: filter by class_broad annotation from NB01
# Fallback: use microglia barcode list file if annotation column is absent
microglia_path = ADATA_DIR / "brain_microglia_allgenes.h5ad"

micro_mask = all_cells_adata.obs["class_broad"] == "Immune"
micro_adata = all_cells_adata[micro_mask].copy()

# Fallback to barcode list if annotation-based subset is empty
if micro_adata.n_obs == 0:
    print("WARNING: class_broad annotation gave 0 microglia; falling back to barcode list")
    micro_barcodes_file = MICROGLIA_DIR / f"microglia_cell_barcodes_{UPSTREAM_NOTEBOOK_TAG}.csv"
    if not micro_barcodes_file.exists():
        raise FileNotFoundError(f"Microglia barcode fallback also missing: {micro_barcodes_file}")
    micro_barcodes_ref = pd.read_csv(micro_barcodes_file)
    micro_barcodes_set = set(micro_barcodes_ref.iloc[:, 0].astype(str).str.strip())
    micro_mask = pd.Index(all_cells_adata.obs_names).isin(micro_barcodes_set)
    micro_adata = all_cells_adata[micro_mask].copy()

if micro_adata.n_obs == 0:
    raise RuntimeError("Microglia subset is empty after all strategies.")

print(f"Microglia: {micro_adata.n_obs:,} cells × {micro_adata.n_vars:,} genes")
print(f"  Samples: {micro_adata.obs['Sample'].value_counts().to_dict()}")

if microglia_path.exists() and not OVERWRITE_OUTPUTS:
    raise FileExistsError(f"Output exists and OVERWRITE_OUTPUTS=False: {microglia_path}")
micro_adata.write_h5ad(microglia_path, compression="gzip")
print(f"Microglia AnnData ready: {microglia_path}")

# ── Astrocyte subset (class_broad == "Astro") ─────────────────────────────────
astrocytes_path = ADATA_DIR / "brain_astrocytes_allgenes.h5ad"

astro_mask = all_cells_adata.obs["class_broad"] == "Astro"
astro_adata = all_cells_adata[astro_mask].copy()

if astro_adata.n_obs == 0:
    print("WARNING: class_broad annotation gave 0 astrocytes; falling back to barcode list")
    astro_barcodes_file = ASTRO_DIR / f"astrocyte_cell_barcodes_{UPSTREAM_NOTEBOOK_TAG}.csv"
    if not astro_barcodes_file.exists():
        raise FileNotFoundError(f"Astrocyte barcode fallback also missing: {astro_barcodes_file}")
    astro_barcodes_ref = pd.read_csv(astro_barcodes_file)
    astro_barcodes_set = set(astro_barcodes_ref.iloc[:, 0].astype(str).str.strip())
    astro_mask = pd.Index(all_cells_adata.obs_names).isin(astro_barcodes_set)
    astro_adata = all_cells_adata[astro_mask].copy()

if astro_adata.n_obs == 0:
    raise RuntimeError("Astrocyte subset is empty after all strategies.")

print(f"\nAstrocytes: {astro_adata.n_obs:,} cells × {astro_adata.n_vars:,} genes")
print(f"  Samples: {astro_adata.obs['Sample'].value_counts().to_dict()}")

if astrocytes_path.exists() and not OVERWRITE_OUTPUTS:
    raise FileExistsError(f"Output exists and OVERWRITE_OUTPUTS=False: {astrocytes_path}")
astro_adata.write_h5ad(astrocytes_path, compression="gzip")
print(f"Astrocyte AnnData ready: {astrocytes_path}")

Microglia: 280 cells × 16,033 genes
  Samples: {'Mock-1': 98, 'OG-3': 53, 'OG-1': 52, 'Mock-2': 38, 'OG-2': 20, 'Mock-3': 19}
Microglia AnnData ready: /media/drive_c/Project_Brain_snRNAseq/Analysis/adatas/brain_microglia_allgenes.h5ad

Astrocytes: 249 cells × 16,033 genes
  Samples: {'Mock-2': 164, 'OG-1': 26, 'Mock-1': 21, 'OG-3': 18, 'Mock-3': 10, 'OG-2': 10}
Astrocyte AnnData ready: /media/drive_c/Project_Brain_snRNAseq/Analysis/adatas/brain_astrocytes_allgenes.h5ad


In [8]:
# -- Humanized microglia export (mouse gene names -> human orthologs) ---------
# Translate micro_adata var_names to human gene symbols via mousipy.
# Mouse-gene-space object (brain_microglia_allgenes.h5ad) is preserved above.
# This humanized object (mouse_microglia_humanized_nb02.h5ad) is what NB03 uses
# for cross-species comparison against the human microglia reference.
#
# Hardening for NB14 compatibility:
# Add a stable gene-symbol contract to all exported AnnData objects:
#   mouse_gene, mouse_symbol_norm, human_gene, human_symbol_norm, has_human_ortholog

humanized_micro_path = ADATA_DIR / "mouse_microglia_humanized_nb02.h5ad"
ortholog_path = MICROGLIA_DIR / "mouse_human_orthologs_nb02.csv"

# Build ortholog map once from a single-cell template (cheap; only var names)
_template = translate(micro_adata[:1, :].copy())
ortholog_map = _template.var[["original_gene_symbol"]].reset_index()
ortholog_map.columns = ["Human_Gene", "Mouse_Gene"]
ortholog_map = (
    ortholog_map.dropna()
    .drop_duplicates(subset=["Mouse_Gene"], keep="first")
    .reset_index(drop=True)
)
mouse_to_human = dict(zip(ortholog_map["Mouse_Gene"], ortholog_map["Human_Gene"]))
print(f"Ortholog map: {len(mouse_to_human):,} mouse -> human gene pairs")

# Save ortholog table
df_ortholog = (
    ortholog_map[["Human_Gene", "Mouse_Gene"]]
    .sort_values(["Mouse_Gene", "Human_Gene"])
    .reset_index(drop=True)
)
if ortholog_path.exists() and not OVERWRITE_OUTPUTS:
    raise FileExistsError(f"Output exists and OVERWRITE_OUTPUTS=False: {ortholog_path}")
df_ortholog.to_csv(ortholog_path, index=False)
print(f"Ortholog table ready: {ortholog_path} ({len(df_ortholog):,} rows)")


def _norm_symbol(s):
    s = str(s)
    return "".join(ch for ch in s if ch.isalnum()).upper()


def _attach_gene_symbol_contract(adata_obj, mapping):
    mouse_genes = adata_obj.var_names.astype(str)
    human_genes = [mapping.get(g, np.nan) for g in mouse_genes]

    adata_obj.var["mouse_gene"] = mouse_genes
    adata_obj.var["mouse_symbol_norm"] = [_norm_symbol(g) for g in mouse_genes]
    adata_obj.var["human_gene"] = human_genes
    adata_obj.var["human_symbol_norm"] = [
        _norm_symbol(g) if pd.notna(g) else "" for g in human_genes
    ]
    adata_obj.var["has_human_ortholog"] = pd.Series(human_genes).notna().to_numpy()
    return adata_obj


# Attach contract fields to canonical outputs and re-save hardened artifacts
all_cells_adata = _attach_gene_symbol_contract(all_cells_adata, mouse_to_human)
micro_adata = _attach_gene_symbol_contract(micro_adata, mouse_to_human)
astro_adata = _attach_gene_symbol_contract(astro_adata, mouse_to_human)

all_cells_adata.uns["gene_symbol_contract"] = {
    "version": "nb02_v1",
    "ortholog_source": str(ortholog_path),
    "mouse_field": "mouse_gene",
    "human_field": "human_gene",
    "mouse_norm_field": "mouse_symbol_norm",
    "human_norm_field": "human_symbol_norm",
    "has_ortholog_field": "has_human_ortholog",
}

if allcells_path.exists() and not OVERWRITE_OUTPUTS:
    raise FileExistsError(f"Output exists and OVERWRITE_OUTPUTS=False: {allcells_path}")
all_cells_adata.write_h5ad(allcells_path, compression="gzip")

if microglia_path.exists() and not OVERWRITE_OUTPUTS:
    raise FileExistsError(f"Output exists and OVERWRITE_OUTPUTS=False: {microglia_path}")
micro_adata.write_h5ad(microglia_path, compression="gzip")

if astrocytes_path.exists() and not OVERWRITE_OUTPUTS:
    raise FileExistsError(f"Output exists and OVERWRITE_OUTPUTS=False: {astrocytes_path}")
astro_adata.write_h5ad(astrocytes_path, compression="gzip")

print("Hardened mouse-gene-space artifacts re-saved with gene symbol contract columns.")

# Apply translation for the humanized microglia export
keep_genes = [g for g in micro_adata.var_names.astype(str) if g in mouse_to_human]
humanized_micro = micro_adata[:, keep_genes].copy()
humanized_micro.var["mouse_gene"] = keep_genes
human_names = [mouse_to_human[g] for g in keep_genes]
humanized_micro.var["human_gene"] = human_names
humanized_micro.var["mouse_symbol_norm"] = [_norm_symbol(g) for g in keep_genes]
humanized_micro.var["human_symbol_norm"] = [_norm_symbol(g) for g in human_names]
humanized_micro.var["has_human_ortholog"] = True
humanized_micro.var_names = pd.Index(human_names)

if humanized_micro.var_names.duplicated().any():
    humanized_micro.var_names_make_unique()

print(
    f"Humanized microglia: {humanized_micro.n_obs:,} cells x "
    f"{humanized_micro.n_vars:,} human-named genes"
)
print(f"  ({micro_adata.n_vars - humanized_micro.n_vars:,} mouse genes dropped; no human ortholog)")
print(f"  var columns: {list(humanized_micro.var.columns)}")

if humanized_micro_path.exists() and not OVERWRITE_OUTPUTS:
    raise FileExistsError(f"Output exists and OVERWRITE_OUTPUTS=False: {humanized_micro_path}")
humanized_micro.write_h5ad(humanized_micro_path, compression="gzip")
print(f"Humanized microglia AnnData ready: {humanized_micro_path}")

100%|██████████| 16033/16033 [00:18<00:00, 869.96it/s]


Found direct orthologs for 14680 genes.
Found multiple orthologs for 278 genes.
Found no orthologs for 607 genes.
Found no index in biomart for 468 genes.


  0%|          | 0/278 [00:00<?, ?it/s]/home/jmk/.pyenv/versions/mlenv/lib/python3.10/site-packages/mousipy/mousipy.py:229: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  var = pd.concat([var, new_row])
/home/jmk/.pyenv/versions/mlenv/lib/python3.10/site-packages/mousipy/mousipy.py:229: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  var = pd.concat([var, new_row])
/home/jmk/.pyenv/versions/mlenv/lib/python3.10/site-packages/mousipy/mousipy.py:229: FutureWarning: The behavior of DataFrame concatena

Ortholog map: 14,782 mouse -> human gene pairs
Ortholog table ready: /media/drive_c/Project_Brain_snRNAseq/Analysis/Microglia_analysis/mouse_human_orthologs_nb02.csv (14,782 rows)


... storing 'human_gene' as categorical
... storing 'human_symbol_norm' as categorical
... storing 'human_gene' as categorical
... storing 'human_symbol_norm' as categorical
... storing 'human_gene' as categorical
... storing 'human_symbol_norm' as categorical


Hardened mouse-gene-space artifacts re-saved with gene symbol contract columns.
Humanized microglia: 280 cells x 14,781 human-named genes
  (1,252 mouse genes dropped; no human ortholog)
  var columns: ['gene_ids', 'feature_types', 'genome', 'mouse_gene', 'mouse_symbol_norm', 'human_gene', 'human_symbol_norm', 'has_human_ortholog']
Humanized microglia AnnData ready: /media/drive_c/Project_Brain_snRNAseq/Analysis/adatas/mouse_microglia_humanized_nb02.h5ad


In [9]:
# Production-readiness checks
checks = []

def add_check(name, condition):
    checks.append({"check": name, "passed": bool(condition)})

# all-cells object
add_check("all-cells non-empty", all_cells_adata.n_obs > 0 and all_cells_adata.n_vars > 0)
add_check("all-cells unique obs_names", all_cells_adata.obs_names.is_unique)
add_check("all-cells unique var_names", all_cells_adata.var_names.is_unique)
add_check("all-cells has counts layer", "counts" in all_cells_adata.layers)
add_check("all-cells has log_norm layer", "log_norm" in all_cells_adata.layers)
add_check(
    "all-cells annotation columns present",
    all(c in all_cells_adata.obs.columns for c in ["Sample", "Condition", "class_name", "class_broad", "subclass_name"]),
)
add_check("all-cells covers all samples", all_cells_adata.obs["Sample"].nunique() == len(df))

# gene symbol contract (for NB14 compatibility)
contract_cols = [
    "mouse_gene",
    "mouse_symbol_norm",
    "human_gene",
    "human_symbol_norm",
    "has_human_ortholog",
]
add_check("all-cells symbol contract columns", all(c in all_cells_adata.var.columns for c in contract_cols))
add_check("all-cells contract metadata in uns", "gene_symbol_contract" in all_cells_adata.uns)

# microglia subset (mouse gene space)
add_check("microglia non-empty", micro_adata.n_obs > 0)
add_check("microglia obs columns", all(c in micro_adata.obs.columns for c in ["Sample", "Condition", "class_broad"]))
add_check("microglia class_broad values", (micro_adata.obs["class_broad"] == "Immune").all())
add_check("microglia symbol contract columns", all(c in micro_adata.var.columns for c in contract_cols))

# humanized microglia
add_check("humanized microglia non-empty", humanized_micro.n_obs > 0)
add_check("humanized microglia unique var_names", humanized_micro.var_names.is_unique)
add_check("humanized microglia has mouse_gene column", "mouse_gene" in humanized_micro.var.columns)
add_check("humanized microglia has human_gene column", "human_gene" in humanized_micro.var.columns)
add_check("ortholog table non-empty", len(df_ortholog) > 0)
add_check("ortholog columns valid", set(["Human_Gene", "Mouse_Gene"]).issubset(df_ortholog.columns))

# astrocyte subset
add_check("astrocyte non-empty", astro_adata.n_obs > 0)
add_check("astrocyte class_broad values", (astro_adata.obs["class_broad"] == "Astro").all())
add_check("astrocyte symbol contract columns", all(c in astro_adata.var.columns for c in contract_cols))

checks_df = pd.DataFrame(checks)
display(checks_df)

failed = checks_df.loc[~checks_df["passed"], "check"].tolist()
if failed:
    raise AssertionError(f"Production checks failed: {failed}")

print("All production checks passed.")

,check,passed
0,all-cells non-empty,True
1,all-cells unique obs_names,True
2,all-cells unique var_names,True
3,all-cells has counts layer,True
4,all-cells has log_norm layer,True
5,all-cells annotation columns present,True
6,all-cells covers all samples,True
7,all-cells symbol contract columns,True
8,all-cells contract metadata in uns,True
9,microglia non-empty,True


All production checks passed.


In [10]:
# Pipeline completion summary
print("=" * 60)
print("PIPELINE COMPLETE - 02 Mouse Data Products")
print("=" * 60)

print(f"\nSamples processed: {len(per_sample_adatas)} / {len(df)}")
print(f"Master reference: {master_ref_path}")

print("\nAll-cells AnnData (brain_allcells_allgenes.h5ad)  -> NB06+")
print(f"  {all_cells_adata.n_obs:,} cells x {all_cells_adata.n_vars:,} genes")
print(f"  Cell types: {all_cells_adata.obs['class_broad'].value_counts().to_dict()}")

print("\nMicroglia - mouse gene space (brain_microglia_allgenes.h5ad)  -> NB03 (if needed in mouse space)")
print(f"  {micro_adata.n_obs:,} cells x {micro_adata.n_vars:,} genes")

print("\nMicroglia - humanized (mouse_microglia_humanized_nb02.h5ad)  -> NB03 cross-species")
print(f"  {humanized_micro.n_obs:,} cells x {humanized_micro.n_vars:,} human-named genes")

print("\nAstrocyte AnnData (brain_astrocytes_allgenes.h5ad)  -> NB06+")
print(f"  {astro_adata.n_obs:,} cells x {astro_adata.n_vars:,} genes")

print("\nGene symbol contract fields added for NB14 compatibility:")
print("  ['mouse_gene', 'mouse_symbol_norm', 'human_gene', 'human_symbol_norm', 'has_human_ortholog']")

print("\nOutput paths:")
print(f"  All cells:          {allcells_path}")
print(f"  Microglia:          {microglia_path}")
print(f"  Humanized microglia:{humanized_micro_path}")
print(f"  Astrocytes:         {astrocytes_path}")
print(f"  Ortholog table:     {ortholog_path}")
print("\n" + "=" * 60)

PIPELINE COMPLETE - 02 Mouse Data Products

Samples processed: 6 / 6
Master reference: /media/drive_c/Project_Brain_snRNAseq/Analysis/Results/Tables/all_cell_barcodes_nb01.csv

All-cells AnnData (brain_allcells_allgenes.h5ad)  -> NB06+
  50,894 cells x 16,033 genes
  Cell types: {'Glut': 26357, 'GABA': 22258, 'Vascular': 1223, 'OPC': 399, 'Immune': 280, 'Astro': 249, 'Dopa': 99, 'Sero': 21, 'OEC': 8}

Microglia - mouse gene space (brain_microglia_allgenes.h5ad)  -> NB03 (if needed in mouse space)
  280 cells x 16,033 genes

Microglia - humanized (mouse_microglia_humanized_nb02.h5ad)  -> NB03 cross-species
  280 cells x 14,781 human-named genes

Astrocyte AnnData (brain_astrocytes_allgenes.h5ad)  -> NB06+
  249 cells x 16,033 genes

Gene symbol contract fields added for NB14 compatibility:
  ['mouse_gene', 'mouse_symbol_norm', 'human_gene', 'human_symbol_norm', 'has_human_ortholog']

Output paths:
  All cells:          /media/drive_c/Project_Brain_snRNAseq/Analysis/adatas/brain_allcells

# References

- Scanpy: Wolf et al. *Genome Biol.* 19, 15 (2018).
- AnnData: Virshup et al. (2021).
- scAR: Sheng et al. *Nat. Methods* 21, 60–68 (2024).